# 🕸️ LangGraph — Stateful Agent Orchestration

> *"LangChain to build fast, LangGraph to control deeply."*

**Part of the LangChain stack** · Reached **v1.0** in late 2025 · Used by Klarna, Uber, J.P. Morgan, LinkedIn

---

## Table of Contents
1. [What is LangGraph?](#1)
2. [Core Idea: Agents as Graphs](#2)
3. [Key Concepts](#3)
4. [Architecture](#4)
5. [How a LangGraph Agent Runs](#5)
6. [Conceptual Code](#6)
7. [How LangGraph is Advanced vs Other Frameworks](#7)
8. [Strengths, Weaknesses & When to Use](#8)


<a id="1"></a>
## 1. What is LangGraph?

**LangGraph** is a **low-level orchestration framework and runtime** for building stateful, long-running AI agents. It is the agent engine inside the broader **LangChain** ecosystem.

The relationship matters:

- **LangChain** = high-level abstractions → ship an agent in a few lines.
- **LangGraph** = low-level primitives → fine-grained control over *every* step, branch, and loop.

> In **LangChain 1.0**, agents are now *built on top of LangGraph*. You start with LangChain's simple `create_agent()` and "drop down" to LangGraph when you need custom control — without rewriting.

LangGraph models an agent as a **graph**: nodes do work, edges decide what happens next, and a shared **state** object flows through the whole thing.


<a id="2"></a>
## 2. Core Idea: Agents as Graphs

Most frameworks model an agent as a *loop*. LangGraph models it as a **directed graph (a state machine)**.

```
        ┌─────────┐
   ┌───▶│  Agent  │ (LLM decides: answer or use a tool?)
   │    └────┬────┘
   │         │
   │   ┌─────┴─────┐
   │   ▼           ▼
   │ ┌──────┐   ┌──────┐
   │ │ Tool │   │ END  │  (final answer)
   │ └──┬───┘   └──────┘
   │    │
   └────┘  (loop back with the observation)
```

Why a graph instead of a loop?
- **Cycles** → the ReAct loop (think → act → observe → repeat) is just an edge pointing back.
- **Branches** → conditional edges route to different nodes based on state.
- **Determinism** → you can see and test the exact control flow, instead of trusting a hidden "manager."


<a id="3"></a>
## 3. Key Concepts

| Concept | What it is |
|---|---|
| **State** | A shared, typed object (usually a `TypedDict`) passed to every node. Each node returns updates that get merged in. |
| **Node** | A Python function (or LLM call) that reads state and returns a partial update. The unit of work. |
| **Edge** | A connection between nodes. **Normal edges** always go A→B; **conditional edges** pick the next node based on state. |
| **Checkpointer** | Saves state after every step → **durable persistence**. Restart mid-run and resume exactly where you left off. |
| **Interrupt / Human-in-the-loop** | Pause the graph, wait for a human to review/approve/edit, then continue. |
| **Reducers** | Rules for *how* state updates merge (e.g. *append* to a message list vs *overwrite*). |

### The 2026 headline feature: **Deep Agents**
One **main agent** handles planning and delegates to **sub-agents** for specific tasks, with access to a virtual **file system** for scratch work — enabling long-horizon, multi-step jobs that don't lose the thread.


<a id="4"></a>
## 4. Architecture

```
┌──────────────────────────────────────────────────────────┐
│                    LangChain Stack (2026)                 │
│                                                           │
│   LangChain        →  high-level: create_agent(), chains  │
│   ───────────────────────────────────────────────────    │
│   LangGraph        →  graph runtime: nodes/edges/state    │
│   LangGraph Platform → deploy long-running agents (cloud) │
│   LangSmith        →  observability, tracing, evals       │
└──────────────────────────────────────────────────────────┘
```

- **Durable state** — execution persists automatically; servers can restart mid-conversation.
- **Native streaming** — token-by-token *and* step-by-step, so you can show the agent reasoning live.
- **One framework, many topologies** — single agent, multi-agent, or hierarchical all use the same primitives.


<a id="5"></a>
## 5. How a LangGraph Agent Runs

```
1. User goal enters as initial STATE
        │
        ▼
2. Agent NODE runs → LLM decides next action
        │
   ┌────┴─────────────┐
   ▼                  ▼
3a. Conditional edge  3b. Conditional edge
    → "tool"              → "END"
        │                  │
        ▼                  ▼
4. Tool NODE executes   Return final state
        │
        ▼
5. CHECKPOINTER saves state  (resume-able)
        │
        ▼
6. Edge loops back to Agent NODE  ──► repeat
```

Every transition is explicit and inspectable — that's the whole point.


<a id="6"></a>
## 6. Conceptual Code

```python
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver
from typing import TypedDict, Annotated
from operator import add

# 1. Define the shared state
class AgentState(TypedDict):
    messages: Annotated[list, add]   # reducer: append, don't overwrite

# 2. Define nodes (units of work)
def agent_node(state: AgentState):
    response = llm.invoke(state["messages"])   # LLM reasons
    return {"messages": [response]}

def tool_node(state: AgentState):
    result = run_tool(state["messages"][-1])   # execute a tool
    return {"messages": [result]}

# 3. Conditional routing
def should_continue(state: AgentState):
    last = state["messages"][-1]
    return "tools" if last.tool_calls else END

# 4. Wire the graph
graph = StateGraph(AgentState)
graph.add_node("agent", agent_node)
graph.add_node("tools", tool_node)
graph.add_edge(START, "agent")
graph.add_conditional_edges("agent", should_continue)
graph.add_edge("tools", "agent")     # loop back = the ReAct cycle

# 5. Compile WITH persistence
app = graph.compile(checkpointer=MemorySaver())

# Run — thread_id lets you pause & resume this exact conversation
app.invoke({"messages": [("user", "Research and summarize X")]},
           config={"configurable": {"thread_id": "user-123"}})
```

> The **LangChain 1.0** shortcut: `from langchain.agents import create_agent` builds this same graph for you in one line — then you can still drop down to the graph above when you need control.


<a id="7"></a>
## 7. How LangGraph is Advanced vs Other Frameworks

| Dimension | LangGraph | CrewAI | AutoGen | OpenAI / Claude SDKs |
|---|---|---|---|---|
| **Control model** | **Explicit graph** (you own every edge) | Role-based crews (high abstraction) | Conversation / graph workflows | Single-vendor agent loop |
| **State & persistence** | **Durable checkpointing** built-in; resume mid-run | Basic memory | State + memory in AgentChat | Sessions / context mgmt |
| **Cycles & branching** | First-class (it's a graph) | Sequential/hierarchical processes | Event-driven | Loop is mostly internal |
| **Human-in-the-loop** | **Native interrupts** at any node | Limited | Supported | Built-in approval gates |
| **Observability** | Deep via **LangSmith** | Crew traces | OTEL | Vendor tracing |
| **Lock-in** | Model-agnostic, open | Model-agnostic | Model-agnostic | Tied to one provider |

### What makes it "advanced"
1. **Lowest-level control of any mainstream framework** — nothing is hidden behind a "manager agent." If an agent misbehaves, you can point to the exact node/edge.
2. **Durable execution** — the checkpointer means a crashed server resumes the agent *exactly* where it stopped. Most frameworks lose the run.
3. **One mental model scales** from a single ReAct agent to a hierarchical multi-agent system — you never switch frameworks as complexity grows.
4. **Production stack included** — LangGraph Platform (deploy) + LangSmith (observe/eval) make it the closest thing to a "default" enterprise agent stack in 2026.


<a id="8"></a>
## 8. Strengths, Weaknesses & When to Use

### ✅ Strengths
- Maximum control + transparency (testable, debuggable graphs)
- Durable, resumable, long-running agents
- Mature ecosystem (LangSmith, LangGraph Platform, huge integration library)
- Scales from simple to hierarchical multi-agent without a rewrite

### ⚠️ Weaknesses
- **Steeper learning curve** — you think in nodes, edges, reducers, and state
- More boilerplate than CrewAI for simple "team of agents" tasks
- The wider LangChain ecosystem can feel heavy / fast-moving

### 🎯 Use LangGraph when…
- You need **reliable, stateful, long-running** agents in production
- Control flow is complex (branches, retries, approvals, loops)
- You want first-class observability and the ability to **pause for humans**

> **Rule of thumb:** *LangChain to prototype in minutes, LangGraph to harden for production.*
